In [1]:
import numpy as np
from mc_engine.market.curves import YieldCurve
from mc_engine.market.market_data import EquityMarketData
from mc_engine.processes.gbm import GBMProcess
from mc_engine.processes.heston import HESTONProcess
from mc_engine.processes.variance_gamma import VarianceGammaProcess
from mc_engine.products.equity.basket import BasketOption
from mc_engine.products.equity.european import EuropeanOption
from mc_engine.products.equity.american import AmericanOption
from mc_engine.products.equity.asian import AsianOption

from mc_engine.market.market_data import FIMarketData
from mc_engine.processes.vasicek import VasicekProcess
from mc_engine.processes.cir import CIRProcess
from mc_engine.processes.gaussian_two_factors import G2ppProcess
from mc_engine.products.fixed_income.zcb import ZeroCouponBond
from mc_engine.products.fixed_income.bond_option import BondOption
from mc_engine.products.fixed_income.swaption import Swaption

from mc_engine.pricing.engine import MonteCarloEngine
from mc_engine.pricing.config import MCConfig

In [2]:
# Marktdaten
curve = YieldCurve(
    tenors = np.array([0.25, 0.5, 1.0, 2.0, 5.0]),
    rates  = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
)
mkt = EquityMarketData(spot=100.0, div_yield=0.01, curve=curve)
mkt_fi = FIMarketData(r_spot=0.03,r_forward=np.array([0.3]*10),forward_tenors=np.array([0.3*i for i in range(10)]))

In [5]:
# Prozess und Produkt
process = GBMProcess(mkt, vol=0.20)
option  = EuropeanOption(
    underlying = "AAPL",
    strike     = 100.0,
    maturity   = 1.0,
    is_call    = False
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=200_000, n_steps=252))
result = engine.price(option, {"AAPL": process}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     8.4807
Std Error: 0.0239
95% CI:    [8.4339, 8.5275]


In [6]:
mkt_AAPL = EquityMarketData(spot=100.0, div_yield=0.02, curve=curve)
mkt_MMM = EquityMarketData(spot=90.0, div_yield=0.02, curve=curve)
# Prozess und Produkt
process_AAPL = GBMProcess(mkt, vol=0.20)
process_MMM = GBMProcess(mkt, vol=0.15)
option  = BasketOption(
    underlying = ["AAPL","MMM"],
    strike     = 100.0,
    maturity   = 1.0,
    is_call    = False,
    arithmetic = True
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=20_000, n_steps=252,corrolation_matrix=np.array([[1,0.5],[0.5,1]])))
result = engine.price(option, {"AAPL": process_AAPL,"MMM":process_MMM}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     7.8703
Std Error: 0.0709
95% CI:    [7.7313, 8.0093]


In [7]:
# Prozess und Produkt
process = GBMProcess(mkt, vol=0.20)
option  = AmericanOption(
    underlying = "AAPL",
    strike     = 90.0,
    maturity   = 1.0,
    is_call    = False
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=200_000, n_steps=252))
result = engine.price(option, {"AAPL": process}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     12.8431
Std Error: 0.0303
95% CI:    [12.7838, 12.9025]


In [8]:
# Prozess und Produkt
process = GBMProcess(mkt, vol=0.20)
option  = AsianOption(
    underlying = "AAPL",
    strike     = 100.0,
    maturity   = 1.0,
    is_call    = False
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=200_000, n_steps=252))
result = engine.price(option, {"AAPL": process}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     4.8738
Std Error: 0.0143
95% CI:    [4.8458, 4.9018]


In [9]:
# Prozess und Produkt
process = HESTONProcess(mkt,0.20,0.1,0.1,0.1,0.1)
option  = EuropeanOption(
    underlying = "AAPL",
    strike     = 100.0,
    maturity   = 1.0,
    is_call    = False
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=200_00, n_steps=252))
result = engine.price(option, {"AAPL": process}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     17.8552
Std Error: 0.1426
95% CI:    [17.5757, 18.1346]


In [10]:
mkt_AAPL = EquityMarketData(spot=100.0, div_yield=0.02, curve=curve)
mkt_MMM = EquityMarketData(spot=90.0, div_yield=0.02, curve=curve)
# Prozess und Produkt
process_AAPL = GBMProcess(mkt, vol=0.20)
process_MMM = HESTONProcess(mkt,0.20,0.1,0.1,0.1,0.1)
option  = BasketOption(
    underlying = ["AAPL","MMM"],
    strike     = 100.0,
    maturity   = 1.0,
    is_call    = False,
    arithmetic = True
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=20_000, n_steps=252))
result = engine.price(option, {"AAPL": process_AAPL,"MMM":process_MMM}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     10.1224
Std Error: 0.0845
95% CI:    [9.9568, 10.2881]


In [11]:
# Prozess und Produkt
process = VarianceGammaProcess(mkt, vol=0.3,theta=0.0,nu=0.1)
option  = EuropeanOption(
    underlying = "AAPL",
    strike     = 100.0,
    maturity   = 1.0,
    is_call    = False
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=20_00, n_steps=252))
result = engine.price(option, {"AAPL": process}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     33.7731
Std Error: 0.8329
95% CI:    [32.1405, 35.4056]


In [12]:
process = VasicekProcess(mkt_fi, vol=0.005,kappa=0.5,theta=0.03)
option  = ZeroCouponBond(
    underlying = "Short Rate",
    maturity   = 1.0,
    notional   = 100
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=20000, n_steps=252))
result = engine.price(option, {"Short Rate": process}, curve)

print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     97.0440
Std Error: 0.0017
95% CI:    [97.0408, 97.0473]


In [13]:
process = VasicekProcess(mkt_fi, vol=0.005,kappa=0.5,theta=0.03)
option  = BondOption(
    underlying = "Short Rate",
    notional= 100,
    maturity_option   = 0.0001,
    maturity_bond = 1.0,
    strike = 0
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=20000, n_steps=25))
result = engine.price(option, {"Short Rate": process}, curve)
print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     97.0448
Std Error: 0.0000
95% CI:    [97.0448, 97.0449]


In [14]:
process = CIRProcess(mkt_fi, vol=0.005,kappa=0.5,theta=0.03)
option  = Swaption(
    underlying = "Short Rate",
    notional = 100,
    maturity_option = 1.0, 
    fixed_rate = 0.01,
    freq = 2,
    swap_maturity = 2.0
    
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=20000, n_steps=252))
result = engine.price(option, {"Short Rate": process}, curve)
print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     4.8150
Std Error: 0.0007
95% CI:    [4.8136, 4.8164]


In [15]:
process = VasicekProcess(mkt_fi, vol=0.005,kappa=0.5,theta=0.03)
option  = Swaption(
    underlying = "Short Rate",
    notional = 100,
    maturity_option = 1.0, 
    fixed_rate = 0.01,
    freq = 2,
    swap_maturity = 2.0
    
)

# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=20000, n_steps=252))
result = engine.price(option, {"Short Rate": process}, curve)
print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     3.7774
Std Error: 0.0032
95% CI:    [3.7711, 3.7837]


In [18]:
process = G2ppProcess( curve = curve,
                 a = 0.01, b = 0.01,
                 sigma = 0.005, eta = 0.01,
                 rho = 0.25,
                 x0 = 0.25, y0 = 0.25)
option  = BondOption(
    underlying = "Short Rate",
    notional= 100,
    maturity_option   = 1.0,
    maturity_bond = 2.0,
    strike = 0
)


# Pricing
engine = MonteCarloEngine(MCConfig(n_sims=200, n_steps=252))
result = engine.price(option, {"Short Rate": process}, curve)
print(f"Preis:     {result.price:.4f}")
print(f"Std Error: {result.std_error:.4f}")
print(f"95% CI:    [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")

Preis:     13.6861
Std Error: 0.0166
95% CI:    [13.6536, 13.7187]
